# Anotação assistida no Google Colab

Versão para o Colab do `anotacao/pipeline.py`: importa as fotos de uma pasta como um lote, anota com os anotadores de `anotacao/anotadores.yaml` (o trio do voto: o [Grounding DINO](https://huggingface.co/IDEA-Research/grounding-dino-tiny), o [OWLv2](https://huggingface.co/google/owlv2-base-patch16-ensemble) e o YOLO da equipe, `best.pt`) e vota entre eles. As caixas são **propostas**, gravadas à parte dos rótulos finais: nas imagens de treino, as que os três concordam saem com rótulo, e as outras vão à revisão humana (`revisar.py`); as de teste vão todas à revisão, sem pré-anotação. Detalhes (classes, saída, opções) em [`anotacao/README.md`](https://github.com/vdlwang/ecovision-iterativo/blob/main/anotacao/README.md).

O notebook não copia o código: ele clona o repositório e roda o próprio `pipeline.py`, com as classes de `anotacao/classes.yaml`. Rode as células em ordem.

## 1. GPU

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, escolha uma GPU (a T4 gratuita basta). Sem GPU funciona, só que bem mais devagar.

In [ ]:
!nvidia-smi -L || echo "Sem GPU: a anotação vai rodar na CPU"

## 2. Código

Clona o repositório (ou atualiza, se a célula já rodou) e instala o `ultralytics`, na versão de `anotacao/requirements.txt`, para o anotador `yolo`. O `torch`, o `transformers` e o `scipy` já vêm instalados no Colab.

In [ ]:
RAMO = "main"  # @param {type:"string"}

import os, sys

DIR_REPO = "/content/ecovision-iterativo"
if os.path.isdir(DIR_REPO):
    !git -C {DIR_REPO} fetch -q --depth 1 origin {RAMO} && git -C {DIR_REPO} checkout -q FETCH_HEAD
else:
    !git clone -q --depth 1 -b {RAMO} https://github.com/vdlwang/ecovision-iterativo.git {DIR_REPO}
!git -C {DIR_REPO} log --oneline -1

DIR_ANOTACAO = f"{DIR_REPO}/anotacao"
if DIR_ANOTACAO not in sys.path:
    sys.path.insert(0, DIR_ANOTACAO)

# Só o ultralytics, na versão fixada; o resto do requirements.txt já vem no Colab
with open(f"{DIR_ANOTACAO}/requirements.txt", encoding="utf-8") as f:
    ULTRALYTICS = next(l.split("#")[0].strip() for l in f if l.startswith("ultralytics=="))
!pip install -q {ULTRALYTICS}

## 3. Imagens

Pastas no Google Drive (`/content/drive/MyDrive/...`) sobrevivem ao fim da sessão; pastas fora dele (`/content/...`) somem quando o Colab desconecta. Com *enviar_do_computador*, as fotos escolhidas são copiadas para `pasta_imagens`. No passo 4, as fotos são importadas para `pasta_saida/images` (uma cópia) com o nome `<lote>_<nome>`; importar de novo pula as que já estão lá.

- **conjunto**: com *sorteio*, cada imagem vai para treino ou teste pelo nome (cerca de 1 em cada 10 para o teste, sempre o mesmo para o mesmo nome); *treino* ou *teste* força o de todas as fotos importadas. Depois da importação, o conjunto de uma imagem não muda. As de teste são revisadas sem pré-anotação.
- **regiao**: região de interesse do lote, `x1 y1 x2 y2` normalizada (de 0 a 1); os anotadores descartam as caixas com o centro fora dela. Vazia: a imagem inteira.

**Banco no Drive**: o estado da anotação é um banco SQLite (`anotacoes.sqlite`), que não pode ser usado direto sobre o Drive montado (as travas do SQLite não valem ali, e o banco pode corromper). Com `pasta_saida` no Drive, esta célula copia a saída para `/content/anotacoes`, e o trabalho é feito nessa cópia; o passo 4 a grava de volta no Drive ao terminar. Não rode dois Colabs sobre a mesma `pasta_saida`.

In [ ]:
pasta_imagens = "/content/drive/MyDrive/ecovision/fotos"  # @param {type:"string"}
pasta_saida = "/content/drive/MyDrive/ecovision/anotacoes"  # @param {type:"string"}
enviar_do_computador = False  # @param {type:"boolean"}
lote = "fotos"  # @param {type:"string"}
conjunto = "sorteio"  # @param ["sorteio", "treino", "teste"]
regiao = ""  # @param {type:"string"}
# lote: letras, dígitos, "." e "-"; as imagens ficam como <lote>_<nome> em pasta_saida/images
# regiao: "x1 y1 x2 y2" normalizada, ex.: "0 0.1 1 0.9"; vazia: sem região

import os, shutil, sqlite3
from pathlib import Path

if any(p.startswith("/content/drive") for p in (pasta_imagens, pasta_saida)):
    from google.colab import drive
    drive.mount("/content/drive")


def copiar_saida(origem, destino):
    """Copia a saída; o banco pela API backup do SQLite, num temporário trocado no fim."""
    origem, destino = Path(origem), Path(destino)
    if not origem.exists():
        return
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copytree(origem, destino, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("anotacoes.sqlite*"))
    banco = origem / "anotacoes.sqlite"
    if banco.exists():
        temporario = destino / "anotacoes.sqlite.copiando"
        temporario.unlink(missing_ok=True)
        de, para = sqlite3.connect(banco), sqlite3.connect(temporario)
        try:
            de.backup(para)
        finally:
            de.close()
            para.close()
        for sufixo in ("-wal", "-shm"):
            (destino / f"anotacoes.sqlite{sufixo}").unlink(missing_ok=True)
        os.replace(temporario, destino / "anotacoes.sqlite")


pasta_imagens = Path(pasta_imagens)
pasta_saida_final = Path(pasta_saida)
if str(pasta_saida_final).startswith("/content/drive"):
    pasta_saida = Path("/content/anotacoes")  # cópia de trabalho; o passo 4 grava no Drive
    copiar_saida(pasta_saida_final, pasta_saida)
    print(f"Trabalhando em {pasta_saida}, cópia de {pasta_saida_final}")
else:
    pasta_saida = pasta_saida_final
pasta_imagens.mkdir(parents=True, exist_ok=True)

if enviar_do_computador:
    from google.colab import files
    for nome, conteudo in files.upload().items():
        (pasta_imagens / nome).write_bytes(conteudo)

from visualizar import EXTENSOES_IMAGEM

imagens = sorted(p for p in pasta_imagens.iterdir()
                 if p.is_file() and p.suffix.lower() in EXTENSOES_IMAGEM)
print(f"{len(imagens)} imagem(ns) em {pasta_imagens}")

## 4. Importar, anotar e votar

Roda o `pipeline.py`: importa as fotos de `pasta_imagens` como `lote`, anota as imagens pendentes com cada anotador de `anotadores.yaml`, um de cada vez, e vota. É retomável: as fotos já importadas, as imagens já anotadas por cada anotador e as já votadas são puladas, então, depois de uma desconexão, basta rodar os passos 2 a 4 de novo. Na primeira vez, os pesos do Grounding DINO e do OWLv2 (algumas centenas de MB cada) são baixados do Hugging Face; o `best.pt` vem com o repositório.

Para outra configuração (modelos, limiares, outros anotadores), copie `anotacao/anotadores.yaml` para o Drive, edite a cópia e indique o caminho dela em `config`.

Com `pasta_saida` no Drive, a cópia de trabalho é gravada de volta nele ao fim da célula, mesmo se ela parar por erro ou interrupção. Se o Colab desconectar no meio, o que foi feito desde a última gravação se perde, e o passo 4 o refaz.

In [ ]:
config = ""  # @param {type:"string"}
previa = True  # @param {type:"boolean"}
classes = ""  # @param {type:"string"}
# config vazio: anotacao/anotadores.yaml do repositório (o trio do voto)
# classes vazio: anotacao/classes.yaml do repositório

import pipeline

argv = [str(pasta_imagens), "-o", str(pasta_saida), "--lote", lote]
if conjunto != "sorteio":
    argv += ["--conjunto", conjunto]
if regiao.strip():
    argv += ["--regiao", *regiao.split()]
if config:
    argv += ["--config", config]
if classes:
    argv += ["--classes", classes]
if previa:
    argv.append("--previa")
try:
    pipeline.main(argv)
except ValueError as erro:
    print(f"Erro: {erro}")
except SystemExit:
    pass  # o argparse já imprimiu o motivo
finally:
    if pasta_saida != pasta_saida_final:
        copiar_saida(pasta_saida, pasta_saida_final)
        print(f"Cópia de trabalho gravada em {pasta_saida_final}")

## 5. Conferir

Mostra quantas imagens o voto aprovou, quantas falharam (e por quê) e quantas de teste foram unidas sem classe, e as prévias de um anotador (exige *previa* marcado no passo 4). O `visualizar.html` gerado em `pasta_saida` não abre dentro do Colab: baixe a pasta (passo 6) ou abra pelo Google Drive no computador.

In [ ]:
anotador = "grounding-dino"  # @param ["grounding-dino", "owlv2", "yolo"]
quantas = 6  # @param {type:"integer"}
primeira = 0  # @param {type:"integer"}

from collections import Counter

from IPython.display import Image, display

import saida as formato

estado = formato.ler_estado(pasta_saida)
votos = Counter((l["voto"] or "sem voto") + (f" ({l['motivo']})" if l["motivo"] else "")
                for l in estado.values() if l["status"] == "pendente")
print(f"Pendentes: {sum(votos.values())} de {len(estado)} imagem(ns)")
for voto, quantidade in sorted(votos.items()):
    print(f"  {voto}: {quantidade}")

previas = sorted((pasta_saida / "previa" / anotador).glob("*.jpg"))
print(f"\n{len(previas)} prévia(s) de {anotador}")
for p in previas[primeira:primeira + quantas]:
    print(p.name)
    display(Image(filename=str(p), width=640))

## 6. Salvar e baixar

Com `pasta_saida` no Drive, grava a cópia de trabalho de volta nele (o banco pela API backup do SQLite; o passo 4 já faz isso ao terminar). Depois, compacta a saída num `.zip` e baixa para o computador. Dentro dele, abra `visualizar.html` para navegar pelas anotações, ou rode o `revisar.py` sobre a pasta para revisar as imagens que o voto não aprovou.

In [ ]:
baixar = True  # @param {type:"boolean"}

if pasta_saida != pasta_saida_final:
    copiar_saida(pasta_saida, pasta_saida_final)
    print(f"Salvo em {pasta_saida_final}")

if baixar:
    from google.colab import files

    arquivo = shutil.make_archive(f"/content/{pasta_saida_final.name}", "zip", pasta_saida)
    files.download(arquivo)